In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error


df = pd.read_csv('/kaggle/input/datasets/shambhurajejagadale/student-performance-prediction-dataset/student_dataset_10000_rows.csv')
print("--- Dataset Summary ---")
print(df.info())
print("\n--- Missing Values ---")
print(df.isnull().sum())
print("\n--- Summary Statistics ---")
print(df.describe())



In [ ]:
# Visualizations
plt.figure(figsize=(10, 6))
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

df = df.drop_duplicates()


df['study_attendance_prod'] = df['study_hours'] * df['attendance']
df['rest_study_ratio'] = df['sleep_hours'] / (df['study_hours'] + 1e-5)

X = df.drop(columns=['exam_score'])
y = df['exam_score']

num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

# Polynomial degree (2 = quadratic, 3 = cubic, ...)
DEGREE = 2

# Polynomial expansion BEFORE scaling, so the new features are standardized too
num_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=DEGREE, include_bias=False)),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_features),
        ('cat', OneHotEncoder(drop='first'), cat_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)
print("Features after polynomial expansion:", X_train_scaled.shape[1])

lr_model = LinearRegression()  # linear model on polynomial features = Polynomial Regression
lr_model.fit(X_train_scaled, y_train)


y_pred = lr_model.predict(X_test_scaled)
print("\n--- Scikit-Learn Polynomial Regression Performance ---")
print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error, r2_score

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

class PolynomialRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(PolynomialRegressionModel, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x)

input_dim = X_train_tensor.shape[1]
print(f"Detected input_dim (features after polynomial expansion): {input_dim}")

model = PolynomialRegressionModel(input_dim)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

num_epochs = 1000
train_losses = []

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

model.eval()
with torch.no_grad():
    predictions = model(X_test_tensor).numpy()

print("--- Full-Batch Gradient Descent Performance (Polynomial Regression, degree={}) ---".format(DEGREE))
print(f"MSE: {mean_squared_error(y_test, predictions):.4f}")
print(f"R2 Score: {r2_score(y_test, predictions):.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

X = df["study_hours"].values
y = df["exam_score"].values

# 1. تطبيع البيانات الأساسية (Normalization)
X_norm = (X - np.mean(X)) / np.std(X)
y_norm = (y - np.mean(y)) / np.std(y)

# 2. إضافة ميزة متعددة الحدود (Polynomial Feature: X^2)
X_poly = X_norm**2

# المعاملات (Parameters)
m1 = 0.0  # معامل الدرجة الأولى (X)
m2 = 0.0  # معامل الدرجة الثانية (X^2)
c = 0.0  # نقطة التقاطع (Intercept)

L = 0.01  # معدل التعلم (Learning Rate)
epochs = 100
n = float(len(X_norm))
cost_history = []

for i in range(epochs):
  # معادلة التوقع للدرجة الثانية: y = m1*X + m2*X^2 + c
  y_pred = m1 * X_norm + m2 * X_poly + c

  # حساب التكلفة (MSE)
  cost = (1 / (2 * n)) * np.sum((y_pred - y_norm) ** 2)
  cost_history.append(cost)

  # حساب المشتقات (Gradients) لكل معامل
  D_m1 = (2 / n) * np.sum((y_pred - y_norm) * X_norm)
  D_m2 = (2 / n) * np.sum((y_pred - y_norm) * X_poly)
  D_c = (2 / n) * np.sum(y_pred - y_norm)

  # تحديث المعاملات
  m1 = m1 - L * D_m1
  m2 = m2 - L * D_m2
  c = c - L * D_c

# الرسم البياني
plt.figure(figsize=(12, 5))

# الرسم الأول: البيانات ومنحنى الانحدار المتعدد (مرتب لضمان سلاسة الخط المنحني)
plt.subplot(1, 2, 1)
plt.scatter(X_norm, y_norm, color="blue", alpha=0.3, label="Data Points")

# ترتيب قيم X لضمان ظهور خط الانحدار بشكل منحنى متصل وغير متقطع
sort_idx = np.argsort(X_norm)
X_sorted = X_norm[sort_idx]
y_pred_sorted = m1 * X_sorted + m2 * (X_sorted**2) + c

plt.plot(
    X_sorted,
    y_pred_sorted,
    color="red",
    linewidth=2,
    label="Polynomial Fit (Degree 2)",
)
plt.xlabel("Study Hours (Normalized)")
plt.ylabel("Exam Score (Normalized)")
plt.title("Polynomial Regression using Gradient Descent")
plt.legend()

# الرسم الثاني: انخفاض التكلفة
plt.subplot(1, 2, 2)
plt.plot(range(epochs), cost_history, color="green", linewidth=2)
plt.xlabel("Epochs")
plt.ylabel("Cost (MSE)")
plt.title("Cost Function Reduction over Time")

plt.tight_layout()
plt.show()